# Juntar as tabelas do INE

Junta os CSV do INE (e o da AIMA) num só ficheiro: `data/T06_merged_data.csv`.

## 1. Bibliotecas

In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## 2. Funções para ler os ficheiros do INE

In [2]:
ler_linhas_ine <- function(ficheiro) {
  todas_linhas <- read_lines(ficheiro, locale = locale(encoding = "latin1"))

  # só as linhas que começam com um código de região ("PT: ", "11A: ", ...)
  e_dados <- str_detect(todas_linhas, "^(\\d{4})?;?[0-9A-Z]{1,4}: ")

  read_delim(
    I(todas_linhas[e_dados]),
    delim = ";",
    col_names = FALSE,
    col_types = cols(.default = col_character()),
    show_col_types = FALSE
  )
}

# vírgula decimal -> número
para_numero <- function(x) round(parse_number(x, locale = locale(decimal_mark = ",")), 2)

## 3. Regiões (NUTS III e ilhas)

In [3]:
regioes <- ler_linhas_ine("data/T06_homes_completed.csv") |>
  select(X1) |>
  separate(X1, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  filter(nchar(nuts_code) == 3 | nuts_code %in% c("20", "30"))

regioes

nuts_code,region
<chr>,<chr>
111,Alto Minho
112,Cávado
119,Ave
11A,Área Metropolitana do Porto
11B,Alto Tâmega e Barroso
11C,Tâmega e Sousa
11D,Douro
11E,Terras de Trás-os-Montes
191,Região de Aveiro


## 4. Tabelas com um valor por ano

O `passo` é 5 nas licenças e nos preços de venda (cada ano tem 5 colunas e ficamos com a primeira, o total).

In [4]:
ler_tabela_anual <- function(ficheiro, anos, nome_coluna, passo = 1) {
  colunas_dos_anos <- 2 + (seq_along(anos) - 1) * passo

  ler_linhas_ine(ficheiro) |>
    select(all_of(c(1, colunas_dos_anos))) |>
    set_names(c("region_raw", anos)) |>
    separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
    pivot_longer(-c(nuts_code, region), names_to = "year", values_to = "valor") |>
    mutate(year = as.integer(year), valor = para_numero(valor)) |>
    select(nuts_code, year, valor) |>
    set_names(c("nuts_code", "year", nome_coluna))
}

salario      <- ler_tabela_anual("data/T06_average_wage.csv",      2024:2021, "wage_monthly")
rendimento   <- ler_tabela_anual("data/T06_median_income.csv",     2024:2018, "income_median")
renda        <- ler_tabela_anual("data/T06_median_rent_m2.csv",    2025:2020, "rent_m2")
casas        <- ler_tabela_anual("data/T06_homes_completed.csv",   2025:2011, "homes_completed")
estrangeiros <- ler_tabela_anual("data/T06_foreign_residents.csv", 2023:2021, "foreign_residents")
migracao     <- ler_tabela_anual("data/T06_net_migration.csv",     2025:2021, "net_migration")

licencas <- ler_tabela_anual("data/T06_homes_licensed.csv", 2025:2002, "homes_licensed", passo = 5)
vendas   <- ler_tabela_anual("data/T06_sale_price_m2.csv",  2025:2019, "sale_price_m2",   passo = 5)

# os Açores aparecem duas vezes na renda (20 e 200)
renda <- renda |> filter(nuts_code != "200")

salario
renda

nuts_code,year,wage_monthly
<chr>,<int>,<dbl>
PT,2024,1576.0
PT,2023,1460.8
PT,2022,1362.4
PT,2021,1289.5
1,2024,1582.7
1,2023,1466.7
1,2022,1368.0
1,2021,1294.1
11,2024,1474.7


nuts_code,year,rent_m2
<chr>,<int>,<dbl>
PT,2025,9.29
PT,2024,8.47
PT,2023,7.62
PT,2022,6.88
PT,2021,6.31
PT,2020,5.79
1,2025,9.32
1,2024,8.50
1,2023,7.64


## 5. População por idade

In [5]:
nomes_idades <- c("total", "a0_4", "a5_9", "a10_14", "a15_19", "a20_24", "a25_29", "a30_34",
                  "a35_39", "a40_44", "a45_49", "a50_54", "a55_59", "a60_64", "a65_69",
                  "a70_74", "a75_79", "a80_84", "a85_mais")

populacao_idades <- ler_linhas_ine("data/T06_population_by_age.csv") |>
  mutate(year = as.integer(X1)) |>
  fill(year) |>   # o ano só vem na primeira linha de cada bloco
  select(year, region_raw = X2, X3:X21) |>
  set_names(c("year", "region_raw", nomes_idades)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  mutate(across(total:a85_mais, para_numero))

populacao <- populacao_idades |>
  transmute(
    nuts_code, year,
    population   = total,
    share_0_14   = (a0_4 + a5_9 + a10_14) / total * 100,
    share_15_24  = (a15_19 + a20_24) / total * 100,
    share_25_34  = (a25_29 + a30_34) / total * 100,
    share_35_44  = (a35_39 + a40_44) / total * 100,
    share_45_54  = (a45_49 + a50_54) / total * 100,
    share_55_64  = (a55_59 + a60_64) / total * 100,
    share_65_79  = (a65_69 + a70_74 + a75_79) / total * 100,
    share_80plus = (a80_84 + a85_mais) / total * 100,
    share_20_34  = (a20_24 + a25_29 + a30_34) / total * 100
  ) |>
  mutate(across(starts_with("share_"), ~ round(.x, 2)))

populacao

nuts_code,year,population,share_0_14,share_15_24,share_25_34,share_35_44,share_45_54,share_55_64,share_65_79,share_80plus,share_20_34
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PT,2025,11424031,12.36,10.30,12.80,13.33,14.75,13.13,16.30,7.04,18.24
1,2025,10912573,12.34,10.27,12.84,13.32,14.70,13.06,16.34,7.13,18.27
11,2025,3790554,11.79,10.16,12.19,12.83,14.86,14.16,17.21,6.81,17.56
111,2025,244343,11.41,9.57,10.90,12.25,14.35,13.53,18.83,9.16,15.90
112,2025,449945,12.81,10.92,12.65,13.64,15.47,13.66,15.18,5.68,18.37
119,2025,441329,12.18,10.06,12.15,13.00,15.19,14.64,16.88,5.89,17.46
11A,2025,1861727,11.79,10.11,12.72,13.19,14.98,13.93,16.86,6.41,18.08
11B,2025,83364,9.71,8.14,8.35,10.11,12.89,14.54,23.87,12.40,12.56
11C,2025,416728,12.11,10.91,12.28,12.62,15.12,15.20,16.12,5.64,17.99


## 6. Censos 2021 e parque habitacional 2022

In [6]:
# colunas: total, propriedade, arrendamento, outra situação
regime <- ler_linhas_ine("data/T06_households_tenure_2021.csv") |>
  select(region_raw = X1, households_2021 = X2, rented_households_2021 = X4) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  mutate(across(c(households_2021, rented_households_2021), para_numero)) |>
  select(nuts_code, households_2021, rented_households_2021)

parque <- ler_linhas_ine("data/T06_dwelling_stock_2022.csv") |>
  select(region_raw = X1, dwellings_2022 = X2) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  mutate(dwellings_2022 = para_numero(dwellings_2022)) |>
  select(nuts_code, dwellings_2022)

regime
parque

nuts_code,households_2021,rented_households_2021
<chr>,<dbl>,<dbl>
PT,4142581,922810
1,3962715,891434
11,1379430,306252
111,90671,13534
112,149713,27256
119,153389,34832
11A,681349,186037
11B,35735,3427
11C,148102,25918


nuts_code,dwellings_2022
<chr>,<dbl>
PT,6003809
1,5757767
11,1908900
111,156185
112,202322
119,200498
11A,842440
11B,73457
11C,205637


## 7. Dormidas (de mensal para anual)

Somamos os meses de cada ano e ficamos com 2022 a 2025.

In [7]:
# linha dos meses: ";Agosto de 2026;Julho de 2026;..."
linhas_turismo <- read_lines("data/T06_tourist_nights.csv", locale = locale(encoding = "latin1"))
linha_meses    <- linhas_turismo[str_detect(linhas_turismo, "^;[^;]+ de \\d{4};")][1]

nomes_meses <- str_split(linha_meses, ";")[[1]]
nomes_meses <- nomes_meses[nomes_meses != ""]

dormidas_mensais <- ler_linhas_ine("data/T06_tourist_nights.csv") |>
  select(1:(length(nomes_meses) + 1)) |>
  set_names(c("region_raw", nomes_meses)) |>
  separate(region_raw, into = c("nuts_code", "region"), sep = ": ", extra = "merge") |>
  pivot_longer(-c(nuts_code, region), names_to = "month", values_to = "tourist_nights") |>
  mutate(
    year = as.integer(str_extract(month, "\\d{4}")),
    tourist_nights = para_numero(tourist_nights)
  )

dormidas <- dormidas_mensais |>
  inner_join(regioes |> select(nuts_code), by = "nuts_code") |>
  filter(year >= 2022, year <= 2025) |>
  group_by(nuts_code, year) |>
  summarise(tourist_nights = sum(tourist_nights), .groups = "drop")

dormidas

nuts_code,year,tourist_nights
<chr>,<int>,<dbl>
111,2022,860775
111,2023,928528
111,2024,985240
111,2025,1031079
112,2022,1058649
112,2023,1119300
112,2024,1167899
112,2025,1183221
119,2022,584132


## 8. Estrangeiros residentes em 2024 (AIMA)

Este ficheiro só tem 2024 e está em UTF-8, por isso lemos com `read_csv`.

In [8]:
estrangeiros_aima <- read_csv(
  "data/T06_foreign_residents_aima_2024.csv",
  col_types = cols(nuts_code = col_character()),
  show_col_types = FALSE
) |>
  select(nuts_code, year, foreign_residents_aima_2024)

# 26 regiões, sem repetidas e iguais às da nossa lista
stopifnot(
  nrow(estrangeiros_aima) == 26,
  !anyDuplicated(estrangeiros_aima$nuts_code),
  all(estrangeiros_aima$year == 2024),
  setequal(estrangeiros_aima$nuts_code, regioes$nuts_code)
)

estrangeiros_aima

nuts_code,year,foreign_residents_aima_2024
<chr>,<dbl>,<dbl>
111,2024,20190
112,2024,41510
119,2024,21535
11A,2024,160957
11B,2024,3006
11C,2024,12999
11D,2024,7765
11E,2024,8674
150,2024,167321


## 9. Juntar tudo

Onde não há dados fica `NA`.

In [9]:
esqueleto <- expand_grid(regioes, year = 2021:2024)

unido <- esqueleto |>
  left_join(salario,       by = c("nuts_code", "year")) |>
  left_join(rendimento,    by = c("nuts_code", "year")) |>
  left_join(renda,         by = c("nuts_code", "year")) |>
  left_join(vendas,        by = c("nuts_code", "year")) |>
  left_join(casas,         by = c("nuts_code", "year")) |>
  left_join(licencas,      by = c("nuts_code", "year")) |>
  left_join(estrangeiros,  by = c("nuts_code", "year")) |>
  left_join(populacao,     by = c("nuts_code", "year")) |>
  left_join(migracao,      by = c("nuts_code", "year")) |>
  left_join(dormidas,      by = c("nuts_code", "year")) |>
  left_join(regime,        by = "nuts_code") |>   # repetido em todos os anos
  left_join(parque,        by = "nuts_code") |>   # repetido em todos os anos
  left_join(estrangeiros_aima, by = c("nuts_code", "year")) |>
  arrange(nuts_code, year)

glimpse(unido)

Rows: 104
Columns: 26
$ nuts_code                   <chr> "111", "111", "111", "111", "112", "112", …
$ region                      <chr> "Alto Minho", "Alto Minho", "Alto Minho", …
$ year                        <dbl> 2021, 2022, 2023, 2024, 2021, 2022, 2023, …
$ wage_monthly                <dbl> 1095.0, 1154.3, 1237.7, 1340.1, 1134.3, 11…
$ income_median               <dbl> 9427, 9993, 10755, 11646, 9897, 10474, 112…
$ rent_m2                     <dbl> 4.29, 4.69, 5.35, 6.00, 5.14, 5.77, 6.59, …
$ sale_price_m2               <dbl> 945, 1064, 1108, 1248, 1091, 1230, 1385, 1…
$ homes_completed             <dbl> 588, 596, 581, 656, 1362, 1706, 1465, 1471…
$ homes_licensed              <dbl> 745, 888, 1108, 941, 1754, 2008, 2066, 214…
$ foreign_residents           <dbl> 7505, 9187, 13145, NA, 16781, 19947, 28650…
$ population                  <dbl> 230770, 236830, 241424, 243852, 417412, 43…
$ share_0_14                  <dbl> 11.57, 11.65, 11.58, 11.52, 13.54, 13.54, …
$ share_15_24     

## 10. Verificações

Devem ser 104 linhas, 26 regiões e 0 linhas repetidas.

In [10]:
nrow(unido)
n_distinct(unido$nuts_code)
unido |> count(nuts_code, year) |> filter(n > 1) |> nrow()

unido |>
  summarise(across(everything(), ~ sum(is.na(.x)))) |>
  pivot_longer(everything(), names_to = "column", values_to = "missing")

[1] 104

[1] 26

[1] 0

column,missing
<chr>,<int>
nuts_code,0
region,0
year,0
wage_monthly,0
income_median,0
rent_m2,0
sale_price_m2,0
homes_completed,0
homes_licensed,0


Cada tabela tem as 26 regiões? Tem de dar 0 em todas.

In [11]:
tabelas <- list(
  salario = salario, rendimento = rendimento, renda = renda, vendas = vendas,
  licencas = licencas, estrangeiros = estrangeiros, populacao = populacao,
  migracao = migracao, regime = regime, parque = parque, dormidas = dormidas,
  estrangeiros_aima = estrangeiros_aima
)

for (nome in names(tabelas)) {
  regioes_em_falta <- setdiff(regioes$nuts_code, tabelas[[nome]]$nuts_code)
  cat(nome, ":", length(regioes_em_falta), "regiões em falta\n")
}

# 4 anos de turismo por região (2022-2025)
dormidas |> count(nuts_code) |> filter(n != 4)

salario : 0 regiões em falta
rendimento : 0 regiões em falta
renda : 0 regiões em falta
vendas : 0 regiões em falta
licencas : 0 regiões em falta
estrangeiros : 0 regiões em falta
populacao : 0 regiões em falta
migracao : 0 regiões em falta
regime : 0 regiões em falta
parque : 0 regiões em falta
dormidas : 0 regiões em falta
estrangeiros_aima : 0 regiões em falta


nuts_code,n
<chr>,<int>


Comparação com os ficheiros originais, em 2024.

In [12]:
# 11A: salário 1620.1, renda 9.14, venda 1986, 5804 casas
unido |>
  filter(nuts_code == "11A", year == 2024) |>
  select(nuts_code, year, wage_monthly, rent_m2, sale_price_m2, homes_completed)

# 1A0: população 2419228, migração 56494, 1072111 alojamentos, 31.4% a arrendar, AIMA 574895
# 150: população 577095, AIMA 167321
unido |>
  filter(nuts_code %in% c("1A0", "150"), year == 2024) |>
  transmute(nuts_code, population, net_migration, dwellings_2022,
            rented_share = round(rented_households_2021 / households_2021 * 100, 1),
            foreign_residents_aima_2024)

nuts_code,year,wage_monthly,rent_m2,sale_price_m2,homes_completed
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
11A,2024,1620.1,9.14,1986,5804


nuts_code,population,net_migration,dwellings_2022,rented_share,foreign_residents_aima_2024
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
150,577095,14459,393387,23.0,167321
1A0,2419228,56494,1072111,31.4,574895


### O ranking muda com o rendimento mediano?

Comparamos a ordem das regiões com a correlação de Spearman. Perto de 1 quer dizer que é a mesma.

In [13]:
esforco <- unido |>
  filter(year %in% c(2021, 2024)) |>
  transmute(
    nuts_code, year,
    esforco_salario    = rent_m2 * 80 / wage_monthly * 100,
    esforco_rendimento = rent_m2 * 80 * 12 / income_median * 100
  ) |>
  pivot_wider(names_from = year, values_from = c(esforco_salario, esforco_rendimento))

# ordem em 2024
cor(esforco$esforco_salario_2024, esforco$esforco_rendimento_2024, method = "spearman")

# regiões que pioraram entre 2021 e 2024
cor(esforco$esforco_salario_2024 - esforco$esforco_salario_2021,
    esforco$esforco_rendimento_2024 - esforco$esforco_rendimento_2021, method = "spearman")

[1] 0.9835897

[1] 0.9760684

## 11. Guardar

In [14]:
write_csv(unido, "data/T06_merged_data.csv")